# Tests de integración de `persistencia de flujos/flows`

Notebook orientado a verificar el comportamiento end-to-end de OP-10 `write flows`y OP-11 `read flows`,
considerando resultado tabular, resultado operacional y trazabilidad.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- smoke de imports,
- y configuración básica de display.

Aquí todavía no se prueban escenarios funcionales de `import_trips_from_dataframe()`;
solo se prepara el entorno de trabajo para las secciones siguientes.

### 0.1 Imports generales

Qué prepara: imports base, utilidades de filesystem, PyArrow para inspección parquet y deep copy para no contaminar fixtures.

In [1]:
import copy
import json
import shutil
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

### 0.2 Imports del módulo 

In [2]:
from pylondrina.datasets import FlowDataset
from pylondrina.errors import ExportError
from pylondrina.io.flows import (
    write_flows,
    read_flows,
    WriteFlowsOptions,
    ReadFlowsOptions,
)

### 0.3 Helpers de testing reutilizables

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")

def issue_codes(report) -> list[str]:
    return [issue.code for issue in report.issues]


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def sort_df(df: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    return df.sort_values(by=by).reset_index(drop=True)


def assert_df_equal_untyped(left: pd.DataFrame, right: pd.DataFrame, *, by: list[str]) -> None:
    pd.testing.assert_frame_equal(
        sort_df(left, by),
        sort_df(right, by),
        check_dtype=False,
        check_categorical=False,
    )


def parquet_has_dictionary_encoding(parquet_path: Path, column_name: str) -> bool:
    pf = pq.ParquetFile(parquet_path)
    try:
        names = pf.schema_arrow.names
        idx = names.index(column_name)
        encodings = {
            str(enc).upper()
            for enc in pf.metadata.row_group(0).column(idx).encodings
        }
        return any("DICTIONARY" in enc for enc in encodings)
    finally:
        pf.close()


INTEGRATION_ROOT = Path("./tmp_integration_flows")
ARTIFACTS_ROOT = INTEGRATION_ROOT / "artifacts"


def reset_integration_root() -> Path:
    if INTEGRATION_ROOT.exists():
        shutil.rmtree(INTEGRATION_ROOT)
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    return INTEGRATION_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = ARTIFACTS_ROOT / case_name
    if case_dir.exists():
        shutil.rmtree(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


ORIGINS = [
    "8828308281fffff",
    "8828308283fffff",
    "8828308285fffff",
    "8828308287fffff",
]
DESTINATIONS = [
    "8828308291fffff",
    "8828308293fffff",
    "8828308295fffff",
    "8828308297fffff",
    "8828308299fffff",
]
MODES = ["bus", "metro", "car"]
PURPOSES = ["work", "education", "shopping", "leisure"]
DAY_TYPES = ["weekday", "weekend"]
GENDERS = ["female", "male"]
INCOME_Q = ["1", "3", "5"]
TIME_PERIODS = ["morning_peak", "midday", "afternoon_peak"]


def make_rich_flows_df(*, repeat_blocks: int = 1) -> pd.DataFrame:
    rows = []
    base_ts = pd.Timestamp("2026-04-01T06:00:00Z")

    idx = 0
    for rep in range(repeat_blocks):
        for origin in ORIGINS:
            for destination in DESTINATIONS:
                for mode in MODES:
                    for day_type in DAY_TYPES:
                        for gender in GENDERS:
                            purpose = PURPOSES[idx % len(PURPOSES)]
                            income_q = INCOME_Q[idx % len(INCOME_Q)]
                            time_period = TIME_PERIODS[idx % len(TIME_PERIODS)]

                            window_start = base_ts + pd.Timedelta(hours=(idx % 10)) + pd.Timedelta(days=rep)
                            window_end = window_start + pd.Timedelta(hours=1)

                            flow_count = 5 + (idx % 17)
                            flow_value = round(flow_count * (1.0 + (0.15 if mode == "metro" else 0.05 if mode == "bus" else 0.25)), 3)

                            rows.append(
                                {
                                    "flow_id": f"f_{rep:02d}_{idx:05d}",
                                    "origin_h3_index": origin,
                                    "destination_h3_index": destination,
                                    "flow_count": int(flow_count),
                                    "flow_value": float(flow_value),
                                    "mode": mode,
                                    "purpose": purpose,
                                    "day_type": day_type,
                                    "user_gender": gender,
                                    "income_quintile": income_q,
                                    "time_period": time_period,
                                    "window_start_utc": window_start,
                                    "window_end_utc": window_end,
                                    "avg_trip_weight": round(0.8 + (idx % 9) * 0.21, 3),
                                    "segment_label": f"{mode}|{day_type}|{gender}",
                                }
                            )
                            idx += 1

    return pd.DataFrame(rows)


def make_flow_to_trips_df(flows_df: pd.DataFrame, *, links_per_flow: int = 3) -> pd.DataFrame:
    rows = []
    movement_counter = 0
    for _, row in flows_df.iterrows():
        for _ in range(links_per_flow):
            movement_counter += 1
            rows.append(
                {
                    "flow_id": row["flow_id"],
                    "movement_id": f"m_{movement_counter:07d}",
                }
            )
    return pd.DataFrame(rows)


def make_rich_flowdataset(
    *,
    repeat_blocks: int = 1,
    with_trip_links: bool = False,
    validated: bool = False,
    dataset_id: str = "flow-dset-integration-001",
) -> FlowDataset:
    flows_df = make_rich_flows_df(repeat_blocks=repeat_blocks)
    flow_to_trips_df = make_flow_to_trips_df(flows_df) if with_trip_links else None

    aggregation_spec = {
        "h3_resolution": 8,
        "group_by": ["mode", "day_type", "user_gender"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "min_trips_per_flow": 1,
    }
    metadata = {
        "dataset_id": dataset_id,
        "is_validated": bool(validated),
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-01T12:00:00Z",
                "parameters": {
                    "h3_resolution": 8,
                    "group_by": ["mode", "day_type", "user_gender"],
                    "time_aggregation": "hour",
                    "time_basis": "origin",
                    "min_trips_per_flow": 1,
                },
                "summary": {
                    "n_flows": int(len(flows_df)),
                    "n_trips_in": int(len(flows_df) * 4),
                    "n_trips_aggregated": int(len(flows_df) * 4),
                    "n_trips_dropped": 0,
                    "n_flow_to_trips_rows": int(len(flow_to_trips_df)) if flow_to_trips_df is not None else None,
                },
                "issues_summary": {"counts": {"info": 0, "warning": 0, "error": 0}, "top_codes": []},
            }
        ],
        "notes": {"fixture": "integration_rich_flowdataset"},
    }
    provenance = {
        "derived_from": [
            {
                "source_type": "trips",
                "dataset_id": "trip-dset-origin-001",
                "schema_version": "1.1",
            }
        ],
        "prior_events_summary": {"n_events": 3},
    }

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=aggregation_spec,
        source_trips={"debug": "in_memory_only"},
        metadata=metadata,
        provenance=provenance,
    )


def build_minimal_valid_sidecar(
    dataset: FlowDataset,
    *,
    include_aux: bool,
    storage_format: str = "parquet",
) -> dict:
    return {
        "dataset_type": "flows",
        "format": "golondrina",
        "layout_version": "1.1",
        "storage": {
            "format": storage_format,
            "options": (
                {"compression": "snappy"}
                if storage_format == "parquet"
                else {"compression": "lz4", "version": 2}
            ),
        },
        "dataset_id": dataset.metadata["dataset_id"],
        "artifact_id": "art_minimal_fixture",
        "files": {
            "data": artifact_data_filename(storage_format),
            "metadata": "flows.metadata.json",
            "flow_to_trips": artifact_aux_filename(storage_format) if include_aux else None,
        },
        "aggregation_spec": copy.deepcopy(dataset.aggregation_spec),
        "provenance": copy.deepcopy(dataset.provenance),
        "metadata": copy.deepcopy(dataset.metadata),
        "tables": {
            "flows": {
                "n_rows": int(len(dataset.flows)),
                "n_cols": int(len(dataset.flows.columns)),
                "columns": list(dataset.flows.columns),
            },
            "flow_to_trips": {
                "n_rows": int(len(dataset.flow_to_trips)),
                "n_cols": int(len(dataset.flow_to_trips.columns)),
                "columns": list(dataset.flow_to_trips.columns),
            } if include_aux and dataset.flow_to_trips is not None else None,
        },
    }

def artifact_data_filename(storage_format: str) -> str:
    if storage_format == "parquet":
        return "flows.parquet"
    if storage_format == "feather":
        return "flows.feather"
    raise ValueError(f"storage_format no soportado: {storage_format!r}")


def artifact_aux_filename(storage_format: str) -> str:
    if storage_format == "parquet":
        return "flow_to_trips.parquet"
    if storage_format == "feather":
        return "flow_to_trips.feather"
    raise ValueError(f"storage_format no soportado: {storage_format!r}")

reset_integration_root()

flowdataset_small = make_rich_flowdataset(
    repeat_blocks=1,
    with_trip_links=False,
    validated=False,
    dataset_id="flow-dset-small-001",
)
flowdataset_with_trip_links = make_rich_flowdataset(
    repeat_blocks=1,
    with_trip_links=True,
    validated=False,
    dataset_id="flow-dset-links-001",
)
artifact_dir_temp = ARTIFACTS_ROOT
flowdataset_sidecar_minimal = build_minimal_valid_sidecar(flowdataset_small, include_aux=False)

print("INTEGRATION_ROOT =", INTEGRATION_ROOT.resolve())
print("artifact_dir_temp =", artifact_dir_temp.resolve())
print("flowdataset_small.flows.shape =", flowdataset_small.flows.shape)
print("flowdataset_with_trip_links.flows.shape =", flowdataset_with_trip_links.flows.shape)
print("flowdataset_with_trip_links.flow_to_trips.shape =", flowdataset_with_trip_links.flow_to_trips.shape)
display(flowdataset_small.flows.head(3))
show_ok("Bloque 0 - setup y fixtures ricas para integración")

INTEGRATION_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_integration_flows
artifact_dir_temp = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_integration_flows\artifacts
flowdataset_small.flows.shape = (240, 15)
flowdataset_with_trip_links.flows.shape = (240, 15)
flowdataset_with_trip_links.flow_to_trips.shape = (720, 2)


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female


OK - Bloque 0 - setup y fixtures ricas para integración


### 0.4 Configuración de display

In [4]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Bloque 1 - write feliz con FlowDataset rica

Qué prueba: camino principal correcto de write_flows, side effects en disco, summary, parameters, evento y sidecar formal.

In [5]:
case_dir = make_case_dir("case_01_write_happy")
artifact_path = case_dir / "flows_write_happy"

flows = copy.deepcopy(flowdataset_small)
flows_before = flows.flows.copy(deep=True)
source_trips_before = copy.deepcopy(flows.source_trips)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

effective_root = Path(str(artifact_path) + ".golondrina")
sidecar_path = effective_root / "flows.metadata.json"
sidecar = read_json(sidecar_path)

assert report.ok is True
assert effective_root.exists()
assert (effective_root / "flows.parquet").exists()
assert sidecar_path.exists()
assert not (effective_root / "flow_to_trips.parquet").exists()

assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert report.summary["dataset_id"] == flows.metadata["dataset_id"]
assert report.summary["artifact_id"] == flows.metadata["artifact_id"]
assert report.summary["path"] == str(effective_root)
assert set(report.summary["files_written"]) == {"flows.parquet", "flows.metadata.json"}

assert report.parameters["path"] == str(effective_root)
assert report.parameters["mode"] == "error_if_exists"
assert report.parameters["storage_format"] == "parquet"
assert report.parameters["normalize_artifact_dir"] is True
assert report.parameters["write_flow_to_trips"] is False

assert "artifact_id" in flows.metadata
assert flows.source_trips == source_trips_before
pd.testing.assert_frame_equal(flows.flows, flows_before)

assert sidecar["dataset_type"] == "flows"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"
assert sidecar["storage"]["format"] == "parquet"
assert sidecar["files"]["data"] == "flows.parquet"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None
assert sidecar["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["artifact_id"] == flows.metadata["artifact_id"]
assert "source_trips" not in sidecar
assert "source_trips" not in sidecar["metadata"]

event = flows.metadata["events"][-1]
assert event["op"] == "write_flows"
assert event["parameters"] == report.parameters
# Este assert está escrito contra el contrato cerrado.
display(event["summary"] )
display(report.summary)
assert event["summary"] == report.summary
assert "issues_summary" in event

display(report)
show_ok("Bloque 1 - write feliz con FlowDataset rica")

{'n_flows': 240,
 'n_flow_to_trips': None,
 'path': 'tmp_integration_flows\\artifacts\\case_01_write_happy\\flows_write_happy.golondrina',
 'files_written': ['flows.parquet', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-small-001',
 'artifact_id': 'art_8937e584-0fbe-4c58-a32c-3c8a4f618d1e'}

{'n_flows': 240,
 'n_flow_to_trips': None,
 'files_written': ['flows.parquet', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-small-001',
 'artifact_id': 'art_8937e584-0fbe-4c58-a32c-3c8a4f618d1e',
 'path': 'tmp_integration_flows\\artifacts\\case_01_write_happy\\flows_write_happy.golondrina'}

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_flow_to_trips': None, 'files_written': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_8937e584-0fbe-4c58-a32c-3c8a4f618d1e', 'path': 'tmp_integration_flows\\artifacts\\case_01_write_happy\\flows_write_happy.golondrina'}, parameters={'path': 'tmp_integration_flows\\artifacts\\case_01_write_happy\\flows_write_happy.golondrina', 'mode': 'error_if_exists', 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': True, 'write_flow_to_trips': False})

OK - Bloque 1 - write feliz con FlowDataset rica


### Bloque 2 - write con auxiliar y verificación observable de dictionary encoding

Qué prueba: persistencia correcta de flow_to_trips y que el write usa dictionary encoding en campos categóricos de segmentación, además de dejar una comparación visible de tamaño contra una escritura manual sin diccionario.

In [6]:
case_dir = make_case_dir("case_02_write_with_aux_and_dictionary")
artifact_path = case_dir / "flows_with_aux"

flows = make_rich_flowdataset(
    repeat_blocks=20,   # dataset más grande para que la diferencia sea visible
    with_trip_links=True,
    validated=False,
    dataset_id="flow-dset-dict-001",
)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

good_parquet = artifact_path / "flows.parquet"
aux_parquet = artifact_path / "flow_to_trips.parquet"

assert report.ok is True
assert good_parquet.exists()
assert aux_parquet.exists()

assert report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)
assert "flow_to_trips.parquet" in report.summary["files_written"]

# Verificación fuerte de dictionary encoding en columnas categóricas de group_by
assert parquet_has_dictionary_encoding(good_parquet, "mode") is True
assert parquet_has_dictionary_encoding(good_parquet, "day_type") is True
assert parquet_has_dictionary_encoding(good_parquet, "user_gender") is True

# Comparación visible contra un parquet manual sin dictionary encoding
bad_path = case_dir / "flows_no_dictionary_manual.parquet"
table_no_dict = pa.Table.from_pandas(flows.flows.copy(deep=True), preserve_index=False)
pq.write_table(
    table_no_dict,
    bad_path,
    compression="snappy",
    use_dictionary=False,
)

size_good = good_parquet.stat().st_size
size_bad = bad_path.stat().st_size

print("size_good =", size_good)
print("size_bad  =", size_bad)
print("ratio_bad_over_good =", round(size_bad / size_good, 3))

# No lo hago ultra rígido, pero al menos no debería empeorar.
assert size_good <= size_bad

show_ok("Bloque 2 - write con auxiliar y dictionary encoding observable")

size_good = 43223
size_bad  = 68706
ratio_bad_over_good = 1.59
OK - Bloque 2 - write con auxiliar y dictionary encoding observable


### Bloque 3 - write fatal por colisión de destino con mode='error_if_exists'

Qué prueba: caso fatal/precondición de write cuando el bundle ya existe.

In [7]:
case_dir = make_case_dir("case_03_write_collision")
artifact_path = case_dir / "flows_collision"

flows = copy.deepcopy(flowdataset_small)

# Primera escritura exitosa
report_ok = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)
assert report_ok.ok is True
assert (artifact_path / artifact_data_filename("parquet")).exists()

# Segunda escritura al mismo destino debe fallar
raised = None
try:
    write_flows(
        copy.deepcopy(flowdataset_small),
        artifact_path,
        options=WriteFlowsOptions(
            mode="error_if_exists",
            storage_format="parquet",
            normalize_artifact_dir=False,
            write_flow_to_trips=False,
        ),
    )
except Exception as exc:
    raised = exc

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Bloque 3 - write fatal por colisión de destino")

ExportError(message='Falló la promoción del staging al destino final del bundle .golondrina.', code='WRITE_FLOWS.IO.COMMIT_FAILED', details={'path': 'tmp_integration_flows\\artifacts\\case_03_write_collision\\flows_collision', 'artifact': 'flows.parquet, flows.metadata.json', 'mode': 'error_if_exists', 'reason': 'ExportError: El bundle destino ya existe y mode=\'error_if_exists\'; write_flows abortado.\ncode: WRITE_FLOWS.LAYOUT.BUNDLE_EXISTS\ndetails:\n{\'path\': \'tmp_integration_flows\\\\artifacts\\\\case_03_write_collision\\\\flows_collision\',\n \'mode\': \'error_if_exists\',\n \'artifact\': \'flows_collision\',\n \'reason\': \'bundle_already_exists\',\n \'action\': \'abort\'}\nissue: Issue(level=\'error\', code=\'WRITE_FLOWS.LAYOUT.BUNDLE_EXISTS\', message="El bundle destino ya existe y mode=\'error_if_exists\'; write_flows abortado.", field=None, source_field=None, row_count=None, details={\'path\': \'tmp_integration_flows\\\\artifacts\\\\case_03_write_collision\\\\flows_collisio

OK - Bloque 3 - write fatal por colisión de destino


### Bloque 4 - read feliz desde bundle formal rico

Qué prueba: camino principal correcto de read_flows, sidecar obligatorio, fallback .golondrina, reconstrucción de flows, aggregation_spec, provenance, evento y summary.

In [8]:
case_dir = make_case_dir("case_04_read_happy")
artifact_path = case_dir / "flows_read_happy"

flows = copy.deepcopy(flowdataset_small)
write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)
assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,  # sin sufijo; debe usar fallback a .golondrina
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

effective_root = Path(str(artifact_path) + ".golondrina")

assert read_report.ok is True
assert read_report.parameters["path"] == str(effective_root)
assert read_report.parameters["strict"] is False
assert read_report.parameters["keep_metadata"] is True
assert read_report.parameters["read_flow_to_trips"] is False

assert read_report.summary["n_flows"] == len(flows.flows)
assert read_report.summary["n_columns"] == len(flows.flows.columns)
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None
assert set(read_report.summary["files_read"]) == {"flows.parquet", "flows.metadata.json"}

assert_df_equal_untyped(loaded.flows, flows.flows, by=["flow_id"])
assert loaded.aggregation_spec == flows.aggregation_spec
assert loaded.provenance == flows.provenance
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

event = loaded.metadata["events"][-1]
assert event["op"] == "read_flows"
assert event["parameters"] == read_report.parameters
assert event["summary"] == read_report.summary
assert "issues_summary" in event

display(read_report)
show_ok("Bloque 4 - read feliz desde bundle formal rico")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_columns': 15, 'flow_to_trips_loaded': False, 'n_flow_to_trips': None, 'files_read': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_2b20eec1-7b08-4cc0-b3ba-137dd7c81700'}, parameters={'path': 'tmp_integration_flows\\artifacts\\case_04_read_happy\\flows_read_happy.golondrina', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': False})

OK - Bloque 4 - read feliz desde bundle formal rico


### Bloque 5 - read con auxiliar solicitado y existente

Qué prueba: lectura feliz de flow_to_trips cuando fue persistido y se pide cargarlo.

In [9]:
case_dir = make_case_dir("case_05_read_with_aux")
artifact_path = case_dir / "flows_with_aux"

flows = copy.deepcopy(flowdataset_with_trip_links)
write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)
assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert read_report.ok is True
assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)
assert loaded.flow_to_trips is not None

assert_df_equal_untyped(loaded.flows, flows.flows, by=["flow_id"])
assert_df_equal_untyped(loaded.flow_to_trips, flows.flow_to_trips, by=["flow_id", "movement_id"])

display(loaded.flows)
display(loaded.metadata)
display(read_report.summary)
show_ok("Bloque 5 - read con auxiliar solicitado y existente")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,f_00_00235,8828308287fffff,8828308299fffff,19,21.85,metro,leisure,weekend,male,3,midday,2026-04-01 11:00:00+00:00,2026-04-01 12:00:00+00:00,1.01,metro|weekend|male
236,f_00_00236,8828308287fffff,8828308299fffff,20,25.00,car,work,weekday,female,5,afternoon_peak,2026-04-01 12:00:00+00:00,2026-04-01 13:00:00+00:00,1.22,car|weekday|female
237,f_00_00237,8828308287fffff,8828308299fffff,21,26.25,car,education,weekday,male,1,morning_peak,2026-04-01 13:00:00+00:00,2026-04-01 14:00:00+00:00,1.43,car|weekday|male
238,f_00_00238,8828308287fffff,8828308299fffff,5,6.25,car,shopping,weekend,female,3,midday,2026-04-01 14:00:00+00:00,2026-04-01 15:00:00+00:00,1.64,car|weekend|female


{'dataset_id': 'flow-dset-links-001',
 'is_validated': False,
 'events': [{'op': 'build_flows',
   'ts_utc': '2026-04-01T12:00:00Z',
   'parameters': {'h3_resolution': 8,
    'group_by': ['mode', 'day_type', 'user_gender'],
    'time_aggregation': 'hour',
    'time_basis': 'origin',
    'min_trips_per_flow': 1},
   'summary': {'n_flows': 240,
    'n_trips_in': 960,
    'n_trips_aggregated': 960,
    'n_trips_dropped': 0,
    'n_flow_to_trips_rows': 720},
   'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
    'top_codes': []}},
  {'op': 'write_flows',
   'ts_utc': '2026-04-17T02:14:15Z',
   'parameters': {'path': 'tmp_integration_flows\\artifacts\\case_05_read_with_aux\\flows_with_aux',
    'mode': 'error_if_exists',
    'storage_format': 'parquet',
    'parquet_compression': 'snappy',
    'feather_compression': 'lz4',
    'normalize_artifact_dir': False,
    'write_flow_to_trips': True},
   'summary': {'n_flows': 240,
    'n_flow_to_trips': 720,
    'path': 'tmp_int

{'n_flows': 240,
 'n_columns': 15,
 'flow_to_trips_loaded': True,
 'n_flow_to_trips': 720,
 'files_read': ['flows.parquet',
  'flow_to_trips.parquet',
  'flows.metadata.json'],
 'dataset_id': 'flow-dset-links-001',
 'artifact_id': 'art_8d27747e-3bde-4128-aa46-3673989c006b'}

OK - Bloque 5 - read con auxiliar solicitado y existente


### Bloque 6 - read degradado con auxiliar solicitado pero faltante (strict=False)

Qué prueba: warning/degradación controlada cuando el archivo auxiliar fue pedido, pero no está en disco.

In [10]:
case_dir = make_case_dir("case_06_read_missing_aux")
artifact_path = case_dir / "flows_missing_aux"

flows = copy.deepcopy(flowdataset_with_trip_links)
write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)
assert write_report.ok is True
assert (artifact_path / artifact_aux_filename("parquet")).exists()

# Simulo pérdida del auxiliar
(artifact_path / artifact_aux_filename("parquet")).unlink()

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING" in codes
assert loaded.flow_to_trips is None
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None

display(loaded.flows)
display(loaded.metadata)
display(read_report.summary)
display(read_report.issues)
show_ok("Bloque 6 - read degradado con auxiliar solicitado pero faltante")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,f_00_00235,8828308287fffff,8828308299fffff,19,21.85,metro,leisure,weekend,male,3,midday,2026-04-01 11:00:00+00:00,2026-04-01 12:00:00+00:00,1.01,metro|weekend|male
236,f_00_00236,8828308287fffff,8828308299fffff,20,25.00,car,work,weekday,female,5,afternoon_peak,2026-04-01 12:00:00+00:00,2026-04-01 13:00:00+00:00,1.22,car|weekday|female
237,f_00_00237,8828308287fffff,8828308299fffff,21,26.25,car,education,weekday,male,1,morning_peak,2026-04-01 13:00:00+00:00,2026-04-01 14:00:00+00:00,1.43,car|weekday|male
238,f_00_00238,8828308287fffff,8828308299fffff,5,6.25,car,shopping,weekend,female,3,midday,2026-04-01 14:00:00+00:00,2026-04-01 15:00:00+00:00,1.64,car|weekend|female


{'dataset_id': 'flow-dset-links-001',
 'is_validated': False,
 'events': [{'op': 'build_flows',
   'ts_utc': '2026-04-01T12:00:00Z',
   'parameters': {'h3_resolution': 8,
    'group_by': ['mode', 'day_type', 'user_gender'],
    'time_aggregation': 'hour',
    'time_basis': 'origin',
    'min_trips_per_flow': 1},
   'summary': {'n_flows': 240,
    'n_trips_in': 960,
    'n_trips_aggregated': 960,
    'n_trips_dropped': 0,
    'n_flow_to_trips_rows': 720},
   'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
    'top_codes': []}},
  {'op': 'write_flows',
   'ts_utc': '2026-04-17T02:14:15Z',
   'parameters': {'path': 'tmp_integration_flows\\artifacts\\case_06_read_missing_aux\\flows_missing_aux',
    'mode': 'error_if_exists',
    'storage_format': 'parquet',
    'parquet_compression': 'snappy',
    'feather_compression': 'lz4',
    'normalize_artifact_dir': False,
    'write_flow_to_trips': True},
   'summary': {'n_flows': 240,
    'n_flow_to_trips': 720,
    'path': 't

{'n_flows': 240,
 'n_columns': 15,
 'flow_to_trips_loaded': False,
 'n_flow_to_trips': None,
 'files_read': ['flows.parquet', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-links-001',
 'artifact_id': 'art_e23af1c6-84dc-45ef-967f-286538310365'}

[Issue(level='warning', code='READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING', message='Se solicitó cargar flow_to_trips, pero el archivo no existe; la lectura continuará sin auxiliar bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_flows\\artifacts\\case_06_read_missing_aux\\flows_missing_aux', 'read_flow_to_trips': True, 'files_expected': ['flow_to_trips.parquet'], 'files_read': ['flows.parquet', 'flows.metadata.json'], 'reason': 'missing_flow_to_trips_file', 'recovered': True, 'recovery_action': 'omit_missing_flow_to_trips'})]

OK - Bloque 6 - read degradado con auxiliar solicitado pero faltante


### Bloque 7 - read degradado con sidecar incompleto (strict=False)

Qué prueba: recuperación controlada de dataset_id, artifact_id y aggregation_spec cuando el sidecar está incompleto, sin aceptar sidecar ausente.

In [11]:
case_dir = make_case_dir("case_07_read_incomplete_sidecar")
artifact_path = case_dir / "flows_incomplete_sidecar"

flows = copy.deepcopy(flowdataset_small)
write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)
assert write_report.ok is True

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

original_dataset_id = sidecar["dataset_id"]
original_artifact_id = sidecar["artifact_id"]

# Corrompo solo partes recuperables bajo strict=False
sidecar["dataset_id"] = ""
sidecar["artifact_id"] = None
sidecar["aggregation_spec"] = None

sidecar_path.write_text(json.dumps(sidecar, ensure_ascii=False, indent=2), encoding="utf-8")

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.METADATA.DATASET_ID_REGENERATED" in codes
assert "READ_FLOWS.METADATA.ARTIFACT_ID_SET_NONE" in codes
assert "READ_FLOWS.SIDECAR.AGGREGATION_SPEC_DEFAULTED" in codes

assert loaded.metadata["dataset_id"] != original_dataset_id
assert loaded.metadata["dataset_id"] is not None
assert loaded.metadata["artifact_id"] is None
assert loaded.aggregation_spec == {}

display(read_report.issues)
show_ok("Bloque 7 - read degradado con sidecar incompleto y strict=False")

[Issue(level='warning', code='READ_FLOWS.METADATA.DATASET_ID_REGENERATED', message='dataset_id faltaba o era inválido en el sidecar; se regeneró bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_flows\\artifacts\\case_07_read_incomplete_sidecar\\flows_incomplete_sidecar', 'reason': 'invalid_dataset_id', 'recovered': True, 'recovery_action': 'regenerate_dataset_id', 'dataset_id_status': 'regenerated'}),
 Issue(level='warning', code='READ_FLOWS.METADATA.ARTIFACT_ID_SET_NONE', message='artifact_id faltaba o era inválido en el sidecar; se degradó a None bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_flows\\artifacts\\case_07_read_incomplete_sidecar\\flows_incomplete_sidecar', 'reason': 'invalid_artifact_id', 'recovered': True, 'recovery_action': 'set_artifact_id_none', 'artifact_id_status': 'set_none'}),
 Issue(level='warning', code='READ_FLOWS.SIDECAR.AGGREGATION_SPEC_DEFAULTED', mes

OK - Bloque 7 - read degradado con sidecar incompleto y strict=False


### Bloque 8 - layout fatal sin sidecar

Qué prueba: lectura formal debe fallar si falta flows.metadata.json.

In [12]:
case_dir = make_case_dir("case_08_layout_fatal_missing_sidecar")
artifact_path = case_dir / "flows_without_sidecar"
artifact_path.mkdir(parents=True, exist_ok=True)

flowdataset_small.flows.to_parquet(
    artifact_path / "flows.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None
try:
    read_flows(
        artifact_path,
        options=ReadFlowsOptions(
            strict=False,
            keep_metadata=True,
            read_flow_to_trips=False,
        ),
    )
except Exception as exc:
    raised = exc

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Bloque 8 - layout fatal sin sidecar")

ExportError(message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', details={'path': 'tmp_integration_flows\\artifacts\\case_08_layout_fatal_missing_sidecar\\flows_without_sidecar', 'files_expected': ['flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}, issue=Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_flows\\artifacts\\case_08_layout_fatal_missing_sidecar\\flows_without_sidecar', 'files_expected': ['flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}), issues=(Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable s

OK - Bloque 8 - layout fatal sin sidecar


### Bloque 9 - round-trip rico + política is_validated=False post-read

Qué prueba: write/read juntas con fixture rica, comparación estructural y política clave de lectura. Aquí intencionalmente parto con is_validated=True para verificar que read la fuerce a False.

In [13]:
case_dir = make_case_dir("case_09_roundtrip_policy")
artifact_path = case_dir / "flows_roundtrip"

flows = make_rich_flowdataset(
    repeat_blocks=2,
    with_trip_links=True,
    validated=True,   # intencional para probar la política post-read
    dataset_id="flow-dset-roundtrip-001",
)

flows_before = flows.flows.copy(deep=True)
flow_to_trips_before = flows.flow_to_trips.copy(deep=True)
aggregation_before = copy.deepcopy(flows.aggregation_spec)
provenance_before = copy.deepcopy(flows.provenance)
dataset_id_before = flows.metadata["dataset_id"]
display(flows.flows)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)
assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE" in codes

assert_df_equal_untyped(loaded.flows, flows_before, by=["flow_id"])
assert_df_equal_untyped(loaded.flow_to_trips, flow_to_trips_before, by=["flow_id", "movement_id"])

assert loaded.aggregation_spec == aggregation_before
assert loaded.provenance == provenance_before
assert loaded.metadata["dataset_id"] == dataset_id_before
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

display(loaded.flows)
print("write summary:")
display(write_report.summary)
print("read summary:")
display(read_report.summary)

show_ok("Bloque 9 - round-trip rico + política is_validated=False post-read")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,f_01_00475,8828308287fffff,8828308299fffff,21,24.15,metro,leisure,weekend,male,3,midday,2026-04-02 11:00:00+00:00,2026-04-02 12:00:00+00:00,2.27,metro|weekend|male
476,f_01_00476,8828308287fffff,8828308299fffff,5,6.25,car,work,weekday,female,5,afternoon_peak,2026-04-02 12:00:00+00:00,2026-04-02 13:00:00+00:00,2.48,car|weekday|female
477,f_01_00477,8828308287fffff,8828308299fffff,6,7.50,car,education,weekday,male,1,morning_peak,2026-04-02 13:00:00+00:00,2026-04-02 14:00:00+00:00,0.80,car|weekday|male
478,f_01_00478,8828308287fffff,8828308299fffff,7,8.75,car,shopping,weekend,female,3,midday,2026-04-02 14:00:00+00:00,2026-04-02 15:00:00+00:00,1.01,car|weekend|female


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female
3,f_00_00003,8828308281fffff,8828308291fffff,8,8.40,bus,leisure,weekend,male,1,morning_peak,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,1.43,bus|weekend|male
4,f_00_00004,8828308281fffff,8828308291fffff,9,10.35,metro,work,weekday,female,3,midday,2026-04-01 10:00:00+00:00,2026-04-01 11:00:00+00:00,1.64,metro|weekday|female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,f_01_00475,8828308287fffff,8828308299fffff,21,24.15,metro,leisure,weekend,male,3,midday,2026-04-02 11:00:00+00:00,2026-04-02 12:00:00+00:00,2.27,metro|weekend|male
476,f_01_00476,8828308287fffff,8828308299fffff,5,6.25,car,work,weekday,female,5,afternoon_peak,2026-04-02 12:00:00+00:00,2026-04-02 13:00:00+00:00,2.48,car|weekday|female
477,f_01_00477,8828308287fffff,8828308299fffff,6,7.50,car,education,weekday,male,1,morning_peak,2026-04-02 13:00:00+00:00,2026-04-02 14:00:00+00:00,0.80,car|weekday|male
478,f_01_00478,8828308287fffff,8828308299fffff,7,8.75,car,shopping,weekend,female,3,midday,2026-04-02 14:00:00+00:00,2026-04-02 15:00:00+00:00,1.01,car|weekend|female


write summary:


{'n_flows': 480,
 'n_flow_to_trips': 1440,
 'files_written': ['flows.parquet',
  'flows.metadata.json',
  'flow_to_trips.parquet'],
 'dataset_id': 'flow-dset-roundtrip-001',
 'artifact_id': 'art_22b55a2d-62b0-469b-9006-072640fc3612',
 'path': 'tmp_integration_flows\\artifacts\\case_09_roundtrip_policy\\flows_roundtrip'}

read summary:


{'n_flows': 480,
 'n_columns': 15,
 'flow_to_trips_loaded': True,
 'n_flow_to_trips': 1440,
 'files_read': ['flows.parquet',
  'flow_to_trips.parquet',
  'flows.metadata.json'],
 'dataset_id': 'flow-dset-roundtrip-001',
 'artifact_id': 'art_22b55a2d-62b0-469b-9006-072640fc3612'}

OK - Bloque 9 - round-trip rico + política is_validated=False post-read


## Bloque 10 - write feliz con backend default Feather

Qué prueba: que `write_flows` ahora usa Feather por defecto, escribe `flows.feather`, deja sidecar consistente y no muta la tabla principal en memoria.

In [14]:
case_dir = make_case_dir("case_10_write_default_feather")
artifact_path = case_dir / "flows_write_default_feather"

flows = copy.deepcopy(flowdataset_small)
flows_before = flows.flows.copy(deep=True)
source_trips_before = copy.deepcopy(flows.source_trips)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
        feather_compression="uncompressed",
    ),
)

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

assert report.ok is True
assert artifact_path.exists()
assert (artifact_path / artifact_data_filename("feather")).exists()
assert not (artifact_path / artifact_data_filename("parquet")).exists()
assert sidecar_path.exists()
assert not (artifact_path / artifact_aux_filename("feather")).exists()

assert report.parameters["storage_format"] == "feather"
#assert report.parameters["feather_compression"] == "lz4"
assert report.parameters["write_flow_to_trips"] is False

assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert report.summary["dataset_id"] == flows.metadata["dataset_id"]
assert report.summary["artifact_id"] == flows.metadata["artifact_id"]
assert report.summary["path"] == str(artifact_path)
assert set(report.summary["files_written"]) == {"flows.feather", "flows.metadata.json"}

pd.testing.assert_frame_equal(flows.flows, flows_before)
assert flows.source_trips == source_trips_before

assert sidecar["dataset_type"] == "flows"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"
assert sidecar["storage"]["format"] == "feather"
#assert sidecar["storage"]["options"]["compression"] == "lz4"
assert sidecar["storage"]["options"]["version"] == 2
assert sidecar["files"]["data"] == "flows.feather"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None
assert sidecar["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["artifact_id"] == flows.metadata["artifact_id"]

event = flows.metadata["events"][-1]
assert event["op"] == "write_flows"
assert event["parameters"] == report.parameters
assert event["summary"] == report.summary
assert "issues_summary" in event

display(report)
display(sidecar)
show_ok("Bloque 10 - write feliz con backend default Feather")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_flow_to_trips': None, 'files_written': ['flows.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_2e521d5d-1c1c-4311-a6f3-108e1ab9eee9', 'path': 'tmp_integration_flows\\artifacts\\case_10_write_default_feather\\flows_write_default_feather'}, parameters={'path': 'tmp_integration_flows\\artifacts\\case_10_write_default_feather\\flows_write_default_feather', 'mode': 'error_if_exists', 'storage_format': 'feather', 'parquet_compression': 'snappy', 'feather_compression': 'uncompressed', 'normalize_artifact_dir': False, 'write_flow_to_trips': False})

{'dataset_type': 'flows',
 'format': 'golondrina',
 'layout_version': '1.1',
 'storage': {'format': 'feather',
  'options': {'compression': 'uncompressed', 'version': 2}},
 'dataset_id': 'flow-dset-small-001',
 'artifact_id': 'art_2e521d5d-1c1c-4311-a6f3-108e1ab9eee9',
 'files': {'data': 'flows.feather',
  'metadata': 'flows.metadata.json',
  'flow_to_trips': None},
 'aggregation_spec': {'h3_resolution': 8,
  'group_by': ['mode', 'day_type', 'user_gender'],
  'time_aggregation': 'hour',
  'time_basis': 'origin',
  'min_trips_per_flow': 1},
 'provenance': {'derived_from': [{'source_type': 'trips',
    'dataset_id': 'trip-dset-origin-001',
    'schema_version': '1.1'}],
  'prior_events_summary': {'n_events': 3}},
 'metadata': {'dataset_id': 'flow-dset-small-001',
  'is_validated': False,
  'events': [{'op': 'build_flows',
    'ts_utc': '2026-04-01T12:00:00Z',
    'parameters': {'h3_resolution': 8,
     'group_by': ['mode', 'day_type', 'user_gender'],
     'time_aggregation': 'hour',
    

OK - Bloque 10 - write feliz con backend default Feather


## Bloque 11 - read feliz desde bundle Feather

Qué prueba: reconstrucción correcta desde un bundle Feather formal, incluyendo fallback a `.golondrina`, `files_read`, metadata/evento y política `is_validated=False`.

In [15]:
case_dir = make_case_dir("case_11_read_happy_feather")
artifact_path = case_dir / "flows_read_happy_feather"

flows = copy.deepcopy(flowdataset_small)
write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)
assert write_report.ok is True

loaded, read_report = read_flows(
    artifact_path,  # sin sufijo; debe usar fallback a .golondrina
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

effective_root = Path(str(artifact_path) + ".golondrina")

assert read_report.ok is True
assert read_report.parameters["path"] == str(effective_root)
assert read_report.parameters["strict"] is False
assert read_report.parameters["keep_metadata"] is True
assert read_report.parameters["read_flow_to_trips"] is False

assert read_report.summary["n_flows"] == len(flows.flows)
assert read_report.summary["n_columns"] == len(flows.flows.columns)
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None
assert set(read_report.summary["files_read"]) == {"flows.feather", "flows.metadata.json"}

assert_df_equal_untyped(loaded.flows, flows.flows, by=["flow_id"])
assert loaded.aggregation_spec == flows.aggregation_spec
assert loaded.provenance == flows.provenance
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

event = loaded.metadata["events"][-1]
assert event["op"] == "read_flows"
assert event["parameters"] == read_report.parameters
assert event["summary"] == read_report.summary
assert "issues_summary" in event

display(read_report)
show_ok("Bloque 11 - read feliz desde bundle Feather")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_columns': 15, 'flow_to_trips_loaded': False, 'n_flow_to_trips': None, 'files_read': ['flows.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_8694ecb1-971e-4c9f-bba7-fdff503b9d8d'}, parameters={'path': 'tmp_integration_flows\\artifacts\\case_11_read_happy_feather\\flows_read_happy_feather.golondrina', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': False})

OK - Bloque 11 - read feliz desde bundle Feather


## Bloque 12 - round-trip Feather con auxiliar presente

Qué prueba: write/read completo con backend Feather, incluyendo `flow_to_trips`, igualdad estructural, `files_read`/`files_written` y política `metadata["is_validated"]=False` post-read.

In [16]:
case_dir = make_case_dir("case_12_roundtrip_feather_with_aux")
artifact_path = case_dir / "flows_roundtrip_feather"

flows = make_rich_flowdataset(
    repeat_blocks=2,
    with_trip_links=True,
    validated=True,
    dataset_id="flow-dset-roundtrip-feather-001",
)

flows_before = flows.flows.copy(deep=True)
flow_to_trips_before = flows.flow_to_trips.copy(deep=True)
aggregation_before = copy.deepcopy(flows.aggregation_spec)
provenance_before = copy.deepcopy(flows.provenance)
dataset_id_before = flows.metadata["dataset_id"]

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)
assert write_report.ok is True
assert (artifact_path / "flows.feather").exists()
assert (artifact_path / "flow_to_trips.feather").exists()
assert set(write_report.summary["files_written"]) == {
    "flows.feather",
    "flow_to_trips.feather",
    "flows.metadata.json",
}

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE" in codes
assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)
assert set(read_report.summary["files_read"]) == {
    "flows.feather",
    "flow_to_trips.feather",
    "flows.metadata.json",
}

assert_df_equal_untyped(loaded.flows, flows_before, by=["flow_id"])
assert_df_equal_untyped(loaded.flow_to_trips, flow_to_trips_before, by=["flow_id", "movement_id"])

assert loaded.aggregation_spec == aggregation_before
assert loaded.provenance == provenance_before
assert loaded.metadata["dataset_id"] == dataset_id_before
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

display(write_report.summary)
display(read_report.summary)
show_ok("Bloque 12 - round-trip Feather con auxiliar presente")

{'n_flows': 480,
 'n_flow_to_trips': 1440,
 'files_written': ['flows.feather',
  'flows.metadata.json',
  'flow_to_trips.feather'],
 'dataset_id': 'flow-dset-roundtrip-feather-001',
 'artifact_id': 'art_f71ec464-405d-4aa2-b10d-985a15270330',
 'path': 'tmp_integration_flows\\artifacts\\case_12_roundtrip_feather_with_aux\\flows_roundtrip_feather'}

{'n_flows': 480,
 'n_columns': 15,
 'flow_to_trips_loaded': True,
 'n_flow_to_trips': 1440,
 'files_read': ['flows.feather',
  'flow_to_trips.feather',
  'flows.metadata.json'],
 'dataset_id': 'flow-dset-roundtrip-feather-001',
 'artifact_id': 'art_f71ec464-405d-4aa2-b10d-985a15270330'}

OK - Bloque 12 - round-trip Feather con auxiliar presente


## Bloque 13 - read degradado con auxiliar Feather solicitado pero faltante

Qué prueba: warning/degradación controlada cuando `flow_to_trips.feather` fue pedido pero no está en disco, bajo `strict=False`.

In [17]:
case_dir = make_case_dir("case_13_read_missing_aux_feather")
artifact_path = case_dir / "flows_missing_aux_feather"

flows = copy.deepcopy(flowdataset_with_trip_links)
write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)
assert write_report.ok is True
assert (artifact_path / "flow_to_trips.feather").exists()

# Simulo pérdida del auxiliar Feather
(artifact_path / "flow_to_trips.feather").unlink()

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = issue_codes(read_report)

assert read_report.ok is True
assert "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING" in codes
assert loaded.flow_to_trips is None
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None
assert set(read_report.summary["files_read"]) == {"flows.feather", "flows.metadata.json"}

display(read_report.issues)
display(read_report.summary)
show_ok("Bloque 13 - read degradado con auxiliar Feather solicitado pero faltante")

[Issue(level='warning', code='READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING', message='Se solicitó cargar flow_to_trips, pero el archivo no existe; la lectura continuará sin auxiliar bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_flows\\artifacts\\case_13_read_missing_aux_feather\\flows_missing_aux_feather', 'read_flow_to_trips': True, 'files_expected': ['flow_to_trips.feather'], 'files_read': ['flows.feather', 'flows.metadata.json'], 'reason': 'missing_flow_to_trips_file', 'recovered': True, 'recovery_action': 'omit_missing_flow_to_trips'})]

{'n_flows': 240,
 'n_columns': 15,
 'flow_to_trips_loaded': False,
 'n_flow_to_trips': None,
 'files_read': ['flows.feather', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-links-001',
 'artifact_id': 'art_ffe391cf-c585-4f16-a088-7cefe065bae5'}

OK - Bloque 13 - read degradado con auxiliar Feather solicitado pero faltante


## Bloque 14 - mismatch fatal entre `storage.format="feather"` y `files.data`

Qué prueba: inconsistencia no recuperable entre el backend declarado en sidecar y el archivo principal declarado, usando `strict=True`.

In [18]:
case_dir = make_case_dir("case_14_sidecar_mismatch_feather")
artifact_path = case_dir / "flows_sidecar_mismatch_feather"

flows = copy.deepcopy(flowdataset_small)
write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)
assert write_report.ok is True
assert (artifact_path / "flows.feather").exists()

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

# Fuerzo inconsistencia: sidecar declara backend feather pero apunta al nombre de parquet
sidecar["files"]["data"] = "flows.parquet"

sidecar_path.write_text(
    json.dumps(sidecar, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

raised = None
try:
    read_flows(
        artifact_path,
        options=ReadFlowsOptions(
            strict=True,
            keep_metadata=True,
            read_flow_to_trips=False,
        ),
    )
except Exception as exc:
    raised = exc

assert raised is not None
assert isinstance(raised, ExportError)
assert getattr(raised, "code", None) == "READ_FLOWS.LAYOUT.MISSING_DATA_FILE"

display(raised)
show_ok("Bloque 14 - mismatch fatal entre storage.format y files.data")

ExportError(message='El bundle de flows no contiene o no resuelve correctamente el archivo principal de datos requerido.', code='READ_FLOWS.LAYOUT.MISSING_DATA_FILE', details={'path': 'tmp_integration_flows\\artifacts\\case_14_sidecar_mismatch_feather\\flows_sidecar_mismatch_feather', 'files_expected': ['flows.feather'], 'reason': "data_file_mismatch: declared='flows.parquet' expected='flows.feather'", 'action': 'abort'}, issue=Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_DATA_FILE', message='El bundle de flows no contiene o no resuelve correctamente el archivo principal de datos requerido.', field=None, source_field=None, row_count=None, details={'path': 'tmp_integration_flows\\artifacts\\case_14_sidecar_mismatch_feather\\flows_sidecar_mismatch_feather', 'files_expected': ['flows.feather'], 'reason': "data_file_mismatch: declared='flows.parquet' expected='flows.feather'", 'action': 'abort'}), issues=(Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_DATA_FILE', message='El 

OK - Bloque 14 - mismatch fatal entre storage.format y files.data


## Bloque 15 - write explícito en Parquet sigue funcionando

Qué prueba: que la incorporación de Feather no rompió la ruta explícita de escritura en Parquet, y que el sidecar/summary siguen consistentes con ese backend.

In [19]:
case_dir = make_case_dir("case_15_write_explicit_parquet_still_works")
artifact_path = case_dir / "flows_write_explicit_parquet"

flows = copy.deepcopy(flowdataset_small)
flows_before = flows.flows.copy(deep=True)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

assert report.ok is True
assert artifact_path.exists()
assert (artifact_path / "flows.parquet").exists()
assert not (artifact_path / "flows.feather").exists()
assert sidecar_path.exists()

assert report.parameters["storage_format"] == "parquet"
assert report.parameters["parquet_compression"] == "snappy"
assert report.parameters["write_flow_to_trips"] is False

assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert set(report.summary["files_written"]) == {"flows.parquet", "flows.metadata.json"}

pd.testing.assert_frame_equal(flows.flows, flows_before)

assert sidecar["storage"]["format"] == "parquet"
assert sidecar["files"]["data"] == "flows.parquet"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None

event = flows.metadata["events"][-1]
assert event["op"] == "write_flows"
assert event["parameters"] == report.parameters
assert event["summary"] == report.summary
assert "issues_summary" in event

display(report)
display(sidecar)
show_ok("Bloque 15 - write explícito en Parquet sigue funcionando")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_flow_to_trips': None, 'files_written': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_8ec77767-b0ff-43c4-ad78-8e049aa7e718', 'path': 'tmp_integration_flows\\artifacts\\case_15_write_explicit_parquet_still_works\\flows_write_explicit_parquet'}, parameters={'path': 'tmp_integration_flows\\artifacts\\case_15_write_explicit_parquet_still_works\\flows_write_explicit_parquet', 'mode': 'error_if_exists', 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': False, 'write_flow_to_trips': False})

{'dataset_type': 'flows',
 'format': 'golondrina',
 'layout_version': '1.1',
 'storage': {'format': 'parquet', 'options': {'compression': 'snappy'}},
 'dataset_id': 'flow-dset-small-001',
 'artifact_id': 'art_8ec77767-b0ff-43c4-ad78-8e049aa7e718',
 'files': {'data': 'flows.parquet',
  'metadata': 'flows.metadata.json',
  'flow_to_trips': None},
 'aggregation_spec': {'h3_resolution': 8,
  'group_by': ['mode', 'day_type', 'user_gender'],
  'time_aggregation': 'hour',
  'time_basis': 'origin',
  'min_trips_per_flow': 1},
 'provenance': {'derived_from': [{'source_type': 'trips',
    'dataset_id': 'trip-dset-origin-001',
    'schema_version': '1.1'}],
  'prior_events_summary': {'n_events': 3}},
 'metadata': {'dataset_id': 'flow-dset-small-001',
  'is_validated': False,
  'events': [{'op': 'build_flows',
    'ts_utc': '2026-04-01T12:00:00Z',
    'parameters': {'h3_resolution': 8,
     'group_by': ['mode', 'day_type', 'user_gender'],
     'time_aggregation': 'hour',
     'time_basis': 'origin

OK - Bloque 15 - write explícito en Parquet sigue funcionando


## Bloque 16 - read de bundle Parquet formal sigue funcionando

Qué prueba: compatibilidad de lectura hacia atrás. Aunque Feather sea el backend por defecto de escritura, `read_flows` debe seguir reconstruyendo correctamente un artefacto formal Parquet guiándose por el sidecar.

In [20]:
case_dir = make_case_dir("case_16_read_legacy_formal_parquet")
artifact_path = case_dir / "flows_read_legacy_formal_parquet"

flows = copy.deepcopy(flowdataset_with_trip_links)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)
assert write_report.ok is True
assert (artifact_path / "flows.parquet").exists()
assert (artifact_path / "flow_to_trips.parquet").exists()

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert read_report.ok is True
assert set(read_report.summary["files_read"]) == {
    "flows.parquet",
    "flow_to_trips.parquet",
    "flows.metadata.json",
}
assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)

assert_df_equal_untyped(loaded.flows, flows.flows, by=["flow_id"])
assert_df_equal_untyped(loaded.flow_to_trips, flows.flow_to_trips, by=["flow_id", "movement_id"])

assert loaded.aggregation_spec == flows.aggregation_spec
assert loaded.provenance == flows.provenance
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

event = loaded.metadata["events"][-1]
assert event["op"] == "read_flows"
assert event["parameters"] == read_report.parameters
assert event["summary"] == read_report.summary
assert "issues_summary" in event

display(read_report)
show_ok("Bloque 16 - read de bundle Parquet formal sigue funcionando")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_columns': 15, 'flow_to_trips_loaded': True, 'n_flow_to_trips': 720, 'files_read': ['flows.parquet', 'flow_to_trips.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-links-001', 'artifact_id': 'art_2293d787-1cdc-415a-a7fa-bdc1cf687c6d'}, parameters={'path': 'tmp_integration_flows\\artifacts\\case_16_read_legacy_formal_parquet\\flows_read_legacy_formal_parquet', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': True})

OK - Bloque 16 - read de bundle Parquet formal sigue funcionando


## Bloque 17 - smoke test de segmentación categórica en Feather

Qué prueba: que la ruta de persistencia con columnas de segmentación categórica (`group_by`) funciona correctamente en Feather, sin depender de un benchmark de tamaño/tiempo.

In [21]:
case_dir = make_case_dir("case_17_feather_segmented_categorical_smoke")
artifact_path = case_dir / "flows_feather_segmented_categorical_smoke"

flows = make_rich_flowdataset(
    repeat_blocks=3,
    with_trip_links=False,
    validated=False,
    dataset_id="flow-dset-segmented-categorical-001",
)

# Refuerzo explícito de segmentaciones típicas de baja cardinalidad
n = len(flows.flows)

mode_values = (["walk", "bus", "metro", "walk"] * ((n // 4) + 1))[:n]
gender_values = (["F", "M"] * ((n // 2) + 1))[:n]

flows.flows["mode"] = pd.Series(mode_values, index=flows.flows.index, dtype="string")
flows.flows["user_gender"] = pd.Series(gender_values, index=flows.flows.index, dtype="string")

flows.aggregation_spec = copy.deepcopy(flows.aggregation_spec)
flows.aggregation_spec["group_by"] = ["mode", "user_gender"]

flows_before = flows.flows.copy(deep=True)

write_report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="feather",
        feather_compression="uncompressed",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)
assert write_report.ok is True
assert (artifact_path / "flows.feather").exists()

loaded, read_report = read_flows(
    artifact_path,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

assert read_report.ok is True
assert set(read_report.summary["files_read"]) == {"flows.feather", "flows.metadata.json"}

# Comparación sin exigir dtype idéntico category/string, solo equivalencia lógica
assert_df_equal_untyped(
    loaded.flows.sort_values(["flow_id"]).reset_index(drop=True),
    flows_before.sort_values(["flow_id"]).reset_index(drop=True),
    by=["flow_id"],
)

assert loaded.aggregation_spec["group_by"] == ["mode", "user_gender"]
assert loaded.metadata["is_validated"] is False

display(write_report.summary)
display(read_report.summary)
show_ok("Bloque 17 - smoke test de segmentación categórica en Feather")

{'n_flows': 720,
 'n_flow_to_trips': None,
 'files_written': ['flows.feather', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-segmented-categorical-001',
 'artifact_id': 'art_5ed71cc4-8993-4f2e-b197-372ac07bfc71',
 'path': 'tmp_integration_flows\\artifacts\\case_17_feather_segmented_categorical_smoke\\flows_feather_segmented_categorical_smoke'}

{'n_flows': 720,
 'n_columns': 15,
 'flow_to_trips_loaded': False,
 'n_flow_to_trips': None,
 'files_read': ['flows.feather', 'flows.metadata.json'],
 'dataset_id': 'flow-dset-segmented-categorical-001',
 'artifact_id': 'art_5ed71cc4-8993-4f2e-b197-372ac07bfc71'}

OK - Bloque 17 - smoke test de segmentación categórica en Feather
